# VCC × STATE：一次完整 toy training step

这个 notebook 用**随机初始化的微型模型**演示当前仓库中的训练架构，不加载 SE-600M 或 ST-SE-Tahoe 权重，也不启动正式训练。重点是看清：

1. 不同数据集保留自己的 native gene panel；
2. frozen SE 如何把每个细胞变成固定维度的 `X_state`；
3. 一个 meta-batch 如何同时容纳 H1、K562 和 VCC control Set；
4. 每个张量维度的具体含义；
5. ST、panel-free decoder、loss、forward 和 backward 如何串起来。

本例使用 `B=3` 个 Set、每个 Set `S=4` 个细胞。正式配置通常使用 `S=256`。

## 1. 本例中的三个 Set

```text
Meta-batch (B=3)
├── Set 0: H1 / TP53 / 4 cells / 5 measured genes
├── Set 1: K562 / MYC / 4 cells / 3 measured genes
└── Set 2: VCC context A / non-targeting / 4 cells / 6 measured genes
```

Set 内部细胞属于同一个 dataset、context、batch 和 perturbation。不同 Set 之间不会做 Transformer attention。VCC control Set 的 target 是另一组 control cells，用于较低权重的 identity/context-adaptation loss。

In [9]:
from pathlib import Path
import sys
import torch
import pandas as pd
from IPython.display import display

repo = Path.cwd()
if not (repo / 'src').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'src'))
sys.path.insert(0, str(repo / 'examples'))

from vcc_toy_training import (
    ToyStateModel, build_toy_batch, compute_losses, gradient_norms,
    log_cp10k, make_gene_embeddings, make_toy_sets,
)

torch.manual_seed(23)
print('repository:', repo)
print('torch:', torch.__version__)

repository: /sde/vcc/vcc2026/state
torch: 2.7.1+cu126


In [10]:
sets = make_toy_sets()
summary = pd.DataFrame([
    {
        'set': i, 'dataset': s.dataset, 'context': s.context,
        'perturbation': s.perturbation, 'cells (S)': s.control_counts.shape[0],
        'native genes': len(s.genes), 'gene names': ', '.join(s.genes),
        'latent weight': s.latent_weight,
    }
    for i, s in enumerate(sets)
])
display(summary)
print('H1 control raw counts [S=4, G_H1=5]:')
display(pd.DataFrame(sets[0].control_counts.numpy(), columns=sets[0].genes))
print('H1 TP53 target raw counts:')
display(pd.DataFrame(sets[0].target_counts.numpy(), columns=sets[0].genes))

,set,dataset,context,perturbation,cells (S),native genes,gene names,latent weight
0,0,Arc_H1,H1,TP53,4,5,"TP53, GENEA, GENEB, GENEC, GENED",1.00
1,1,Replogle,K562,MYC,4,3,"MYC, GENEA, GENEX",1.00
2,2,VCC_controls,A,non-targeting,4,6,"TP53, MYC, GENEA, GENEB, GENEX, GENEY",0.25


H1 control raw counts [S=4, G_H1=5]:


,TP53,GENEA,GENEB,GENEC,GENED
0,8.0,2.0,4.0,1.0,5.0
1,7.0,3.0,5.0,1.0,4.0
2,9.0,2.0,3.0,2.0,4.0
3,8.0,4.0,4.0,1.0,3.0


H1 TP53 target raw counts:


,TP53,GENEA,GENEB,GENEC,GENED
0,4.0,2.0,7.0,1.0,6.0
1,3.0,3.0,8.0,1.0,5.0
2,5.0,2.0,6.0,2.0,5.0
3,4.0,4.0,7.0,1.0,4.0


## 2. 模拟 frozen SE

真实流程中，每个细胞的 native-panel raw counts通过 frozen SE-600M 变成固定维度 `X_state`。本例只模拟这个接口：

```text
H1:    [4, 5] raw counts ─┐
K562:  [4, 3] raw counts ─┼─ simulated frozen SE ─→ 各自 [4, 8] X_state
VCC-A: [4, 6] raw counts ─┘
```

这里的 8 是 toy `state_dim`。真实 Tahoe `X_state` 维度由 checkpoint/profile 决定。模拟 SE 投影不参与梯度更新。

In [11]:
all_genes = sorted({gene for cell_set in sets for gene in cell_set.genes})
embedding_lookup = make_gene_embeddings(all_genes, embedding_dim=5)
batch, latent_weights = build_toy_batch(sets, embedding_lookup, state_dim=8)

dimension_meaning = {
    'ctrl_state': '[B=3, S=4, D_state=8]：每个control cell的frozen SE embedding',
    'target_state': '[3, 4, 8]：真实perturbed/control-target cell的SE embedding',
    'pert_embedding': '[B=3, E_pert=5]：每个Set的target-gene语义向量',
    'gene_embeddings': '[B=3, Gmax=6, E_gene=5]：各Set待监督gene，padding到6',
    'gene_targets': '[B=3, S=4, Gmax=6]：log1p(CP10K)监督值',
    'gene_mask': '[3, 4, 6]：True=真实测量，False=padding，不计loss',
}
shape_table = pd.DataFrame([
    {'tensor': key, 'actual shape': tuple(value.shape), 'meaning': dimension_meaning[key]}
    for key, value in batch.items()
])
display(shape_table)
print('latent_weights:', latent_weights.tolist(), '(VCC identity Set权重较低)')

,tensor,actual shape,meaning
0,ctrl_state,"(3, 4, 8)","[B=3, S=4, D_state=8]：每个control cell的frozen SE..."
1,target_state,"(3, 4, 8)","[3, 4, 8]：真实perturbed/control-target cell的SE e..."
2,pert_embedding,"(3, 5)","[B=3, E_pert=5]：每个Set的target-gene语义向量"
3,gene_embeddings,"(3, 6, 5)","[B=3, Gmax=6, E_gene=5]：各Set待监督gene，padding到6"
4,gene_targets,"(3, 4, 6)","[B=3, S=4, Gmax=6]：log1p(CP10K)监督值"
5,gene_mask,"(3, 4, 6)","[3, 4, 6]：True=真实测量，False=padding，不计loss"


latent_weights: [1.0, 1.0, 0.25] (VCC identity Set权重较低)


In [12]:
mask_summary = pd.DataFrame(batch['gene_mask'][:, 0].numpy(), columns=[f'padded_gene_{i}' for i in range(6)])
mask_summary.insert(0, 'Set', ['H1', 'K562', 'VCC-A control'])
display(mask_summary)
print('K562只有3个native genes，所以位置3、4、5为False；这些零不是生物学零表达。')

,Set,padded_gene_0,padded_gene_1,padded_gene_2,padded_gene_3,padded_gene_4,padded_gene_5
0,H1,True,True,True,True,True,False
1,K562,True,True,True,False,False,False
2,VCC-A control,True,True,True,True,True,True


K562只有3个native genes，所以位置3、4、5为False；这些零不是生物学零表达。


## 3. Decoder target：CP10K + log1p

`.X` 仍保留 raw integer counts。只有构造 decoder监督时计算：

$$y_{ig}=\log\left(1+\frac{x_{ig}}{L_i}\times 10000\right),\quad L_i=\sum_{g\in G_d}x_{ig}$$

当前实现中的 $L_i$ 是该数据集完整 native panel 的总和，不是 sampled genes 的总和。真实 K562 后续可优先使用 `obs['UMI_count']` 作为更完整的 library size。

In [13]:
raw = sets[0].target_counts[0]
library_size = raw.sum()
cp10k = raw / library_size * 10_000
logged = log_cp10k(raw.unsqueeze(0)).squeeze(0)
display(pd.DataFrame({
    'gene': sets[0].genes, 'raw count': raw.numpy(),
    'CP10K': cp10k.numpy(), 'log1p(CP10K)': logged.numpy(),
}))
print('full native-panel library size =', float(library_size))
print('CP10K sum =', float(cp10k.sum()))

,gene,raw count,CP10K,log1p(CP10K)
0,TP53,4.0,2000.0,7.601402
1,GENEA,2.0,1000.0,6.908755
2,GENEB,7.0,3500.0,8.160804
3,GENEC,1.0,500.0,6.216606
4,GENED,6.0,3000.0,8.006701


full native-panel library size = 20.0
CP10K sum = 10000.0


## 4. 随机初始化的 toy ST + panel-free decoder

```text
ctrl_state [B,S,8] ─→ basal_encoder ─┐
pert_embedding [B,5] → pert_encoder ─┼→ Transformer（Set内attention）→ predicted_state [B,S,8]
                                      │
gene_embeddings [B,Gmax,5] ───────────┴→ shared gene decoder → expression [B,S,Gmax]
```

Decoder 内没有 `Linear(..., number_of_genes)`，所以 H1/K562/VCC 可以共享参数。

In [14]:
model = ToyStateModel(state_dim=8, perturbation_dim=5, hidden_dim=12)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print(model)
print('trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

ToyStateModel(
  (basal_encoder): Linear(in_features=8, out_features=12, bias=True)
  (perturbation_encoder): Linear(in_features=5, out_features=12, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=12, out_features=12, bias=True)
        )
        (linear1): Linear(in_features=12, out_features=24, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=24, out_features=12, bias=True)
        (norm1): LayerNorm((12,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((12,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.0, inplace=False)
        (dropout2): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (project_out): Linear(in_features=12, out_features=8, bias=True)
  (gene_decoder): PanelFreeGeneDecoder(
    (cell_projection): Linear(in

## 5. 一次 forward

`B=3` 是三个独立 Set，`S=4` 是每个 Set 的细胞数。Transformer 把 `B` 当 batch axis，因此不会让 H1 cell attention 到 K562/VCC cell。

In [15]:
predicted_state, predicted_expression = model(batch)
print('predicted_state:', tuple(predicted_state.shape), '= [B, S, D_state]')
print('predicted_expression:', tuple(predicted_expression.shape), '= [B, S, Gmax]')
print('all predictions finite:', bool(torch.isfinite(predicted_expression).all()))

predicted_state: (3, 4, 8) = [B, S, D_state]
predicted_expression: (3, 4, 6) = [B, S, Gmax]
all predictions finite: True


## 6. Loss 如何计算

### Latent distribution loss
对每个 Set 独立计算经验 energy distance：

$$L_{state}^{(b)}=2\,E\lVert\hat h-h\rVert-E\lVert\hat h-\hat h'\rVert-E\lVert h-h'\rVert$$

H1、K562 权重为 1；VCC control identity Set 权重为 0.25，避免 identity任务压制 perturbation effect。

### Gene loss
对每个 Set 的有效 gene位置计算 masked MSE，再对 Set 求平均：

$$L_{gene}^{(b)}=\frac{\sum_{i,g}m_{big}(\hat y_{big}-y_{big})^2}{\sum_{i,g}m_{big}}$$

总损失：$L=L_{state}+0.5L_{gene}$。这里的 0.5 只是 toy 超参数。

In [16]:
losses = compute_losses(
    predicted_state, predicted_expression, batch, latent_weights, gene_loss_weight=0.5
)
loss_table = pd.DataFrame({
    'Set': ['H1/TP53', 'K562/MYC', 'VCC-A/non-targeting'],
    'latent energy': losses['latent_per_set'].detach().numpy(),
    'masked gene MSE': losses['gene_per_set'].detach().numpy(),
    'latent weight': latent_weights.numpy(),
})
display(loss_table)
print('weighted latent loss =', float(losses['latent_loss']))
print('mean gene loss      =', float(losses['gene_loss']))
print('total loss          =', float(losses['total_loss']))

,Set,latent energy,masked gene MSE,latent weight
0,H1/TP53,5.521594,47.432304,1.00
1,K562/MYC,6.381556,51.499554,1.00
2,VCC-A/non-targeting,3.459192,45.093899,0.25


weighted latent loss = 5.674643516540527
mean gene loss      = 48.00858688354492
total loss          = 29.678936004638672


## 7. 一次 backward 和 optimizer step

下面严格走一遍训练 step：

```text
optimizer.zero_grad()
→ forward
→ total_loss.backward()
→ 检查各模块梯度
→ optimizer.step()
```

模拟 SE 不属于 `model.parameters()`，因此不会得到梯度。

In [17]:
optimizer.zero_grad(set_to_none=True)
predicted_state, predicted_expression = model(batch)
losses = compute_losses(predicted_state, predicted_expression, batch, latent_weights, gene_loss_weight=0.5)

weight_before = model.project_out.weight.detach().clone()
losses['total_loss'].backward()
norms = gradient_norms(model)
display(pd.DataFrame({'module': norms.keys(), 'gradient L2 norm': norms.values()}))
assert all(value > 0 for value in norms.values())

optimizer.step()
parameter_update = (model.project_out.weight.detach() - weight_before).norm().item()
print('project_out parameter update L2 norm =', parameter_update)
assert parameter_update > 0
print('一次完整 forward + backward + optimizer.step 已成功。')

,module,gradient L2 norm
0,basal_encoder,2.675258
1,perturbation_encoder,1.541119
2,transformer,4.014446
3,gene_decoder,7.574692


project_out parameter update L2 norm = 0.009800302796065807
一次完整 forward + backward + optimizer.step 已成功。


## 8. Toy shape 到正式训练 shape 的对应关系

| 项目 | Toy | 正式训练示例 |
|---|---:|---:|
| Set 数 `B` | 3 | `training.batch_size=4` |
| 每个 Set 细胞数 `S` | 4 | `cell_set_len=256` |
| 一个 step 的 cell 数 | 12 | `4×256=1024` |
| state dim | 8 | Tahoe checkpoint 的 `X_state` dim |
| semantic gene dim | 5 | 本地 SE embeddings 为 5120 |
| sampled genes | 最多 6 | 默认最多 1024/Set |

正式数据流程还需要先完成 metadata标准化、QC、gene alias映射和 frozen SE transform。VCC A/B/C controls可以作为低权重 identity Set、decoder监督和 library-size校准，但不能提供 TP53/MYC 等 perturbation-response标签。